# TUGAS PRAKTIKUM 3 KECERDASAN KOMPUTASIONAL

| Informasi Mahasiswa | |
| :--- | :--- |
| **Nama** | Catherine Aprilia Harlina |
| **NRP** | 5054251012 |
| **Program Studi** | Rekayasa Kecerdasan Artifisial |
| **Mata Kuliah** | Kecerdasan Komputasional |

---

In [1]:
# =============================================================================
# Standard setup cell.
# Run this once, before any other cell in this notebook.
# =============================================================================
import importlib.util
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/kcv-if/Modul-Praktikum-KK-RKA-25.git"
ON_COLAB = "google.colab" in sys.modules

def ensure_dependencies():
    """Install only the packages this module actually uses."""
    required = {
        "networkx": "networkx",
        "numpy": "numpy",
        "pandas": "pandas",
        "matplotlib": "matplotlib",
        "ipywidgets": "ipywidgets",
        "PIL": "pillow",
        "pygments": "pygments",
    }
    missing = [pkg for mod, pkg in required.items() if importlib.util.find_spec(mod) is None]
    if missing:
        print("Installing:", ", ".join(missing))
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)

def find_environment(start):
    """Locate the folder that holds logic.py and utils.py, searching upward."""
    for root in [start, *start.parents]:
        for candidate in sorted(root.rglob("logic.py")):
            if (candidate.parent / "utils.py").exists():
                return candidate.parent
        if (root / ".git").exists():
            break
    return None

ensure_dependencies()

start_dir = Path.cwd()
if ON_COLAB:
    clone_dir = Path("Modul-Praktikum-KK-RKA-25")
    if not clone_dir.exists():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(clone_dir)], check=True)
    start_dir = clone_dir

ENV_DIR = find_environment(start_dir)
if ENV_DIR is None:
    raise RuntimeError(
        "Environment folder not found. Make sure this notebook is opened from "
        "inside the Modul-Praktikum-KK-RKA-25 repository."
    )

if str(ENV_DIR) not in sys.path:
    sys.path.insert(0, str(ENV_DIR))

import itertools
import warnings
import pandas as pd

warnings.filterwarnings("ignore", message="no QP solver found")

from logic import *
from notebook import psource
from utils import *

print("Environment :", ENV_DIR)
print("Python      :", sys.version.split()[0])
print("Check       : tt_entails(P & Q, Q) =", tt_entails(expr("P & Q"), expr("Q")))

Environment : c:\Users\LEGION\Praktikum-KK\logics\praktikum\environment
Python      : 3.12.4
Check       : tt_entails(P & Q, Q) = True


1: analisis klasifikasi syntax buat ekspresi AdvisorOf(Ani) nah kategori dari ekspresi ini tuh masuknya ke Term alias aplikasi fungsi dmn arity nya tuh 1 trs kl variabel bebasnya tuh gak ada krn Ani disini berkedudukan sbg constant kan nah kl alasan knp dia gak bisa dinilai benar atau salah tuh krn AdvisorOf(Ani) cuma merujuk atau ngerefer ke satu entitas individu spesifik doang yaitu siapa dosen pembimbingnya Ani dan dia tuh bukan berupa pernyataan atau klaim utuh makanya gak punya nilai kebenaran alias True False gitu.

2: nyatain analisis buat ekspresi Lecturer(AdvisorOf(Ani)) nah bentuknya tuh berupa Formula Atomik dmn arity nya terdiri dari Predikat Lecturer ber-arity 1 trs di dalamnya ada Fungsi AdvisorOf yg ber-arity 1 juga trs kl variabel bebasnya tuh gak ada sama sekali nah makanya status dari ekspresi ini tuh tergolong sbg Sentence krn emang gak mengandung free variable di dalamnya kek gitu.

3: intinya nyatain analisis buat ekspresi Takes(x, AI) nah bentuk dari ekspresi ini tuh berupa Formula Atomik dmn arity nya tuh Predikat Takes ber-arity 2 trs kl variabel bebasnya tuh ada si x krn dia diawali huruf kecil sesuai konvensi AIMA atau logic.py nah makanya statusnya tuh tergolong sbg Open Formula krn memuat variabel bebas x yg emang belum diikat sm quantifier apapun kek gitu.

In [11]:
# verifikasi syntax soal 1
exprs = [expr('AdvisorOf(Ani)'), expr('Lecturer(AdvisorOf(Ani))'), expr('Takes(x, AI)')]
for e in exprs:
    print(f"expr: {str(e):25} | op: {str(e.op):10} | args: {str(e.args):20} | free vars: {variables(e)}")

expr: AdvisorOf(Ani)            | op: AdvisorOf  | args: (Ani,)               | free vars: set()
expr: Lecturer(AdvisorOf(Ani))  | op: Lecturer   | args: (AdvisorOf(Ani),)    | free vars: set()
expr: Takes(x, AI)              | op: Takes      | args: (x, AI)              | free vars: {x}


In [14]:
# verifikasi evaluasi model soal 2
d = {'Ani', 'Budi', 'Cici'}
student, diligent = {'Ani', 'Budi'}, {'Budi', 'Cici'}

# 1. universal
all_sd = all((x not in student) or (x in diligent) for x in d)
ce = [x for x in d if x in student and x not in diligent]
print(f"1. ∀x (Student(x) ⇒ Diligent(x))\n   Hasil        : {all_sd}\n   Counterexample: {ce} (Ani mahasiswa tp ga rajin)\n")

# 2. existential
some_sd = any(x in student and x in diligent for x in d)
witness = [x for x in d if x in student and x in diligent]
print(f"2. ∃x (Student(x) ∧ Diligent(x))\n   Hasil        : {some_sd}\n   Witness      : {witness} (Budi mahasiswa sekaligus rajin)")

1. ∀x (Student(x) ⇒ Diligent(x))
   Hasil        : False
   Counterexample: ['Ani'] (Ani mahasiswa tp ga rajin)

2. ∃x (Student(x) ∧ Diligent(x))
   Hasil        : True
   Witness      : ['Budi'] (Budi mahasiswa sekaligus rajin)


1: analisis buat formula ∀x (Student(x) ∧ Smart(x)) nah makna salahnya tuh seolah-olah ngartiin kl semua entitas atau object yg ada di dalam seluruh domain tuh kudu beneran mahasiswa sekaligus pinter tanpa terkecuali trs kl analisis masalahnya tuh timbul grgr pake operator konjungsi (∧) jadinya misal di dalam domain kita ada object yg jelas-jelas bukan mahasiswa contohnya kek Kursi nah nilai dari Student(Kursi) kan otomatis jd False kan trs grgr dia pake ∧ ya akhirannya bikin seluruh formula universal ini langsung bernilai False padahal maksud kita cuma mau ngomongin populasi mahasiswa doang nah makanya formula yg bener tuh kudu diganti pake implikasi jd ∀x (Student(x) ⇒ Smart(x)) kek gitu.

2: intinya nyatain analisis buat formula ∃x (Student(x) ⇒ Smart(x)) nah makna salahnya tuh timbul grgr sifat dasar dari tabel kebenaran implikasi P ⇒ Q yg nilainya tuh bakal otomatis bernilai True tiap kali premis P nya bernilai False trs kl analisis masalahnya tuh misal di dalam domain kita ada object yg bukan mahasiswa kek Kursi tadi nah pas diuji Student(Kursi) bernilai False kan nah ini tuh malah ngebikin implikasi Student(Kursi) ⇒ Smart(Kursi) langsung dapet nilai True scr otomatis padahal Kursi kan ga ada hubungannya ama mahasiswa or pinter nah fenomena ini tuh disebut vacuous witness yg ngebikin klaim keberadaan eksistensialnya jd terpenuhi scr salah atau agak ngawur gitu lah makanya formula yg bener tuh kudu diganti pake konjungsi jd ∃x (Student(x) ∧ Smart(x)) kek gitu.

In [19]:
# test model domain kecil buat soal 3
domain_test = {'Ani', 'Kursi'}
student_test = {'Ani'}
smart_test = set()

# 1. universal comparison
wrong_univ = all(x in student_test and x in smart_test for x in domain_test)
correct_univ = all(x not in student_test or x in smart_test for x in domain_test)

# 2. existential comparison
wrong_exist = any(x not in student_test or x in smart_test for x in domain_test)
correct_exist = any(x in student_test and x in smart_test for x in domain_test)

print("--- PENGUJIAN DOMAIN {'Ani', 'Kursi'} ---")
print(f"Universal Salah (∀x Student(x) ∧ Smart(x))  : {wrong_univ}")
print(f"Universal Benar (∀x Student(x) ⇒ Smart(x))  : {correct_univ}\n")
print(f"Existential Salah (∃x Student(x) ⇒ Smart(x)): {wrong_exist} (Vacuously True krn Kursi bukan Student)")
print(f"Existential Benar (∃x Student(x) ∧ Smart(x)): {correct_exist}")

--- PENGUJIAN DOMAIN {'Ani', 'Kursi'} ---
Universal Salah (∀x Student(x) ∧ Smart(x))  : False
Universal Benar (∀x Student(x) ⇒ Smart(x))  : False

Existential Salah (∃x Student(x) ⇒ Smart(x)): True (Vacuously True krn Kursi bukan Student)
Existential Benar (∃x Student(x) ∧ Smart(x)): False


1: setiap dosen tuh ngajar sedikitnya satu mahasiswa nah fol formal nya tuh ∀x (Lecturer(x) ⇒ ∃y (Student(y) ∧ Teaches(x, y))) trs kl perbedaan scope ama urutan quantifier nya tuh posisi ∀x kan ditaro di paling luar sebelum ∃y nah ini tuh implikasinya penentuan mahasiswa (y) dilakukan setelah kita milih dosen (x) nya dulu makanya mahasiswa yg diajar bisa beda-beda buat tiap dosen tergantung dosen mana yg lg dibahas kek gitu.

2: ada satu mahasiswa spesifik yg diajar sm seluruh dosen nah fol formal nya tuh ∃y (Student(y) ∧ ∀x (Lecturer(x) ⇒ Teaches(x, y))) trs kl perbedaan scope ama urutan quantifier nya tuh balik krn posisi ∃y skrg yg ditaro di paling luar sebelum ∀x nah ini tuh implikasinya kita ngunci atau nentuin satu individu mahasiswa (y) nya dulu dari awal baru abis itu dikaitkan sm semua dosen (x) yg ada makanya dapet arti kl emang ada satu mahasiswa spesifik yg sama persis dan dia tuh jd murid dari seluruh dosen yg ada kek gitu.

In [20]:
# Model Domain
lecturers = {'PakBudi', 'BuRina'}
students = {'Ani', 'Cici'}
# Pengajaran: PakBudi mengajar Ani, BuRina mengajar Cici
teaches_rel = {('PakBudi', 'Ani'), ('BuRina', 'Cici')}
# Kalimat 1: ∀x (Lecturer(x) ⇒ ∃y (Student(y) ∧ Teaches(x,y)))
cond1 = all(
    any((d, m) in teaches_rel for m in students)
    for d in lecturers
)
# Kalimat 2: ∃y (Student(y) ∧ ∀x (Lecturer(x) ⇒ Teaches(x,y)))
cond2 = any(
    all((d, m) in teaches_rel for d in lecturers)
    for m in students
)
print("1. ∀x ∃y (Setiap dosen mengajar minimal 1 mahasiswa) :", cond1)
print("2. ∃y ∀x (Ada 1 mahasiswa yang diajar SELUA dosen)    :", cond2)

1. ∀x ∃y (Setiap dosen mengajar minimal 1 mahasiswa) : True
2. ∃y ∀x (Ada 1 mahasiswa yang diajar SELUA dosen)    : False


kalimat 1: intinya nyatain kl semua mahasiswa yg ngambil matkul ai dan lulus prog brhk ikut praktikum ai nah fol formal nya tuh ∀x ((Student(x) ∧ Takes(x, AI) ∧ Passed(x, Prog)) ⇒ Eligible(x, AILab)) trs variabel bebasnya gak ada kl variabel terikatnya tuh {x} nah folkb encoding nya ditulis kek gini `(Student(x) & Takes(x, AI) & Passed(x, Prog)) ==> Eligible(x, AILab)` kurang lebih itu jawaban k1

kalimat 2: intinya nyatain kl ada mahasiswa yg ngambil matkul ai nah fol formal nya tuh ∃x (Student(x) ∧ Takes(x, AI)) trs variabel bebasnya gak ada kl variabel terikatnya cuma {x} nah kl penjelasan fresh constant nya tuh krn folkb gak mendukung quantifier ∃ scr langsung dtl makanya pake teknik skolemization pake fresh constant contohnya `S_AI` dinamain fresh constant krn simbolnya emang kudu baru dan blm pernah dipake di KB biar gak bentrok makna ama object lain yg udah ada trs folkb encoding nya tinggal dipecah jd `Student(S_AI)` ama `Takes(S_AI, AI)` kek gitu.

In [6]:
# 1. Definisikan Clause
rule_eligible = expr('(Student(x) & Takes(x, AI) & Passed(x, Prog)) ==> Eligible(x, AILab)')
fact_student = expr('Student(S_AI)')
fact_takes = expr('Takes(S_AI, AI)')

# 2. Construct FolKB
my_kb = FolKB([
    rule_eligible,
    fact_student,
    fact_takes
])

print("=== DAFTAR CLAUSE DALAM FOLKB ===")
for i, clause in enumerate(my_kb.clauses, 1):
    print(f"Clause {i}          : {clause}")
    print(f"Is Definite Clause? : {is_definite_clause(clause)}")
    print("-" * 55)

=== DAFTAR CLAUSE DALAM FOLKB ===
Clause 1          : (((Student(x) & Takes(x, AI)) & Passed(x, Prog)) ==> Eligible(x, AILab))
Is Definite Clause? : True
-------------------------------------------------------
Clause 2          : Student(S_AI)
Is Definite Clause? : True
-------------------------------------------------------
Clause 3          : Takes(S_AI, AI)
Is Definite Clause? : True
-------------------------------------------------------
